# Notebook 01 — Dataset Sources and Description

**Thesis**: *Employment in Europe During the AI Transition: Education, Digital Readiness, Gender, Family, Migration, and Institutional Effects*

**Purpose of this notebook**

This notebook describes every dataset used in this thesis: what it is, why it is used, what it covers, and what its limitations are. It is a **description notebook**, not a download or engineering notebook. Technical access details (API calls, manual download steps) are kept short and are not the focus.

**Scope (see `instructions.md` §7)**
- Explain each dataset: source, purpose, country coverage, years, key variables, limitations.
- Show file paths and confirm whether each raw file is present.
- Build the dataset inventory table.
- Keep technical download/API notes short and secondary.
- **Do not** perform EDA, clean data, or run any model in this notebook (see notebooks 02–07).

## Environment Setup

Run the cell below **once** if you get a `ModuleNotFoundError` (for example,
`No module named 'pandas'`). It installs every package listed in
`requirements.txt` into the currently selected Jupyter kernel — skip it if the
packages are already installed. Make sure the correct kernel/virtual environment
is selected in VS Code (top-right of the notebook toolbar) before running any
other cell. Full setup instructions: [docs/environment_setup.md](../docs/environment_setup.md).

In [1]:
# OPTIONAL: run only if a package is missing (e.g. ModuleNotFoundError: No module named 'pandas').
# Safe to skip if all packages in requirements.txt are already installed in this kernel.
from pathlib import Path

_project_root_for_install = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
_requirements_path = _project_root_for_install / 'requirements.txt'

if not _requirements_path.exists():
    print(f'requirements.txt not found at {_requirements_path}. See docs/environment_setup.md.')
else:
    %pip install -r "{_requirements_path}"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Imports and path setup for this notebook.
try:
    import pandas as pd
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        f"Missing package: {exc.name}. Run the optional install cell above (or "
        "`pip install -r requirements.txt` in a terminal), then restart the kernel and re-run this cell."
    ) from exc

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
PICKLE_DIR = DATA_DIR / 'pickle'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
REPORTS_DIR = OUTPUTS_DIR / 'reports'
MODELS_DIR = OUTPUTS_DIR / 'models'
DOCS_DIR = PROJECT_ROOT / 'docs'

# Notebook-specific raw data subfolders.
ESS_DIR = RAW_DIR / 'ess'
ESS_MULTILEVEL_DIR = RAW_DIR / 'ess_multilevel'
EUROSTAT_DIR = RAW_DIR / 'eurostat_ai'
OPTIONAL_DIR = RAW_DIR / 'optional_sources'
OUTPUTS_TABLES = TABLES_DIR  # kept for backward compatibility with earlier cells in this notebook

for d in [ESS_DIR, ESS_MULTILEVEL_DIR, EUROSTAT_DIR, OPTIONAL_DIR, PROCESSED_DIR, PICKLE_DIR,
          FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)

Project root: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis


## 1. European Social Survey (ESS) — Main Individual-Level Dataset

**Source**: European Social Survey, Round 11 (ESS11).

**Purpose**: main individual-level dataset for this thesis. It provides the dependent
variable (employment status) and almost all individual-level explanatory variables:
education, digital readiness, gender, family status, migration background, health,
and institutional trust.

**Country coverage**: all countries participating in ESS Round 11 (24 European
countries in the file used here).

**Years**: ESS Round 11 fieldwork was conducted in 2023–2024 (exact dates vary
slightly by country).

**Key expected variables**: employment status, age, gender, education, marital
status, household size, children/household responsibility, migration background,
health, internet use/digital readiness, institutional trust, discrimination,
country, survey design weights.

**Limitations**: cross-sectional (no individual-level panel across rounds), all
measures are self-reported, and population representativeness depends on applying
the survey design weights.

In [3]:
# Short technical note: ESS requires free registration and manual download from the
# ESS Data Portal, so it cannot be retrieved automatically. This cell only checks
# whether the expected ESS11 file is already present in data/raw/ess/. Both the SPSS
# (.sav) and CSV (.csv) integrated file formats are accepted, since the ESS Data
# Portal can provide either — CSV is checked first because that is the format used
# for this project. See data/raw/ess/README.md for the exact download steps.

ess_expected_files = [
    ESS_DIR / 'ESS11.csv',
    ESS_DIR / 'ESS11.sav',
]
ess_path = next((f for f in ess_expected_files if f.exists()), None)

if ess_path is None:
    print('ESS11 file not found yet.')
    print('Expected one of:')
    for f in ess_expected_files:
        print('  -', f)
    print('Download from the ESS Data Portal (registration required) and place the file at one of the paths above. See data/raw/ess/README.md.')
else:
    print('ESS file found:', ess_path)

ESS file found: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\data\raw\ess\ESS11.csv


## 2. ESS Multilevel Data — Country-Level Institutional/Macroeconomic Context

**Source**: ESS Multilevel Data portal (same ESS credentials).

**Purpose**: adds country-level institutional and macroeconomic indicators to
complement the AI adoption variable, so country-level context is not reduced to AI
adoption alone.

**Country coverage**: matched to the ESS round used above.

**Years**: matched to the ESS round used above; some indicators may lag by a year.

**Key expected variables**: unemployment rate, GDP per capita, inequality/Gini,
social expenditure, migration indicators, education indicators, institutional
indicators.

**Limitations**: indicator availability varies by country and year; some countries
may have partial coverage.

In [4]:
# Short technical note: also a manual, registration-required download.
ess_multilevel_expected_files = [
    ESS_MULTILEVEL_DIR / 'ESS_multilevel_data.sav',
    ESS_MULTILEVEL_DIR / 'ESS_multilevel_data.csv',
]
ess_multilevel_path = next((f for f in ess_multilevel_expected_files if f.exists()), None)

if ess_multilevel_path is None:
    print('ESS Multilevel file not found yet.')
    print('Expected one of:')
    for f in ess_multilevel_expected_files:
        print('  -', f)
    print('Download from the ESS Multilevel Data portal and place the file at one of the paths above.')
else:
    print('ESS Multilevel file found:', ess_multilevel_path)

ESS Multilevel file found: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\data\raw\ess_multilevel\ESS_multilevel_data.csv


## 3. Eurostat AI Adoption Dataset — Country-Level AI Context

**Source**: Eurostat enterprise digital economy / ICT usage statistics.

**Purpose**: provides `ai_adoption_enterprises_pct`, the share of enterprises using
at least one AI technology in each country. This is the one country-level AI
variable used to test whether AI adoption changes the strength of individual-level
employment determinants (H2, H5). It is one contextual variable among several, not
the main topic of the thesis.

**Country coverage**: EU/EEA countries reporting to Eurostat.

**Years**: latest available survey wave at the time of download.

**Key variable**: `ai_adoption_enterprises_pct`.

**Limitations**: an enterprise-level proxy, not an individual-level measure of
exposure to AI; year availability may not perfectly align with the ESS round used.

*The cell below only prepares the raw Eurostat AI adoption file for later notebooks
(checks/downloads a CSV into `data/raw/eurostat_ai/`). It does not clean, merge, or
analyze the data — that happens in notebooks 02 and 03.*

In [5]:
# Short technical note: this is the one dataset that can be retrieved automatically,
# via the Eurostat API (eurostat package). If no local file is present, this cell
# tries to automatically download the Eurostat AI adoption-by-enterprises table
# (expected code "isoc_eb_ai" — table codes can change between Eurostat releases,
# so a keyword search of the table of contents is used as a fallback/cross-check).
EUROSTAT_AI_TABLE_CODE = 'isoc_eb_ai'  # Eurostat: enterprises using artificial intelligence technologies

eurostat_files = list(EUROSTAT_DIR.glob('*.csv'))

if eurostat_files:
    print('Eurostat AI adoption file already present:', eurostat_files[0])
else:
    try:
        import eurostat
    except ImportError:
        print('Automatic download not possible: the eurostat package is not installed (pip install eurostat).')
        print(f'1. What failed: could not import the eurostat package.')
        print(f'2. Dataset needed: Eurostat "Enterprises using artificial intelligence technologies" (table code "{EUROSTAT_AI_TABLE_CODE}").')
        print(f'3. Where to place it: download the CSV manually from the Eurostat website and save it under {EUROSTAT_DIR}')
    else:
        try:
            df_ai_download = eurostat.get_data_df(EUROSTAT_AI_TABLE_CODE)
            if df_ai_download is None or df_ai_download.empty:
                raise ValueError(f'Eurostat returned no data for table "{EUROSTAT_AI_TABLE_CODE}"')

            downloaded_path = EUROSTAT_DIR / f'{EUROSTAT_AI_TABLE_CODE}.csv'
            df_ai_download.to_csv(downloaded_path, index=False)
            print(f'Automatically downloaded Eurostat table "{EUROSTAT_AI_TABLE_CODE}" and saved to:', downloaded_path)
            print('Shape:', df_ai_download.shape)
        except Exception as exc:
            print('Automatic download failed.')
            print(f'1. What failed: could not retrieve Eurostat table "{EUROSTAT_AI_TABLE_CODE}" ({exc}).')
            print(f'2. Dataset needed: Eurostat "Enterprises using artificial intelligence technologies" (table code "{EUROSTAT_AI_TABLE_CODE}"; search "artificial intelligence" on the Eurostat website if this code has changed).')
            print(f'3. Where to place it: download the CSV manually and save it under {EUROSTAT_DIR}')

            # Best-effort candidate list from the table of contents, to help find the
            # right code if it has changed since this notebook was written.
            try:
                toc = eurostat.get_toc_df()
                ai_candidates = toc[toc['title'].str.contains('artificial intelligence', case=False, na=False)]
                if not ai_candidates.empty:
                    print('\nCandidate Eurostat tables mentioning artificial intelligence:')
                    display(ai_candidates[['title', 'code']])
            except Exception:
                pass

Eurostat AI adoption file already present: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\data\raw\eurostat_ai\isoc_eb_ai.csv


## 4. Optional Supporting Sources

Not required for the core hypotheses; used for the literature review and, if
needed, extra robustness controls.

- **Eurofound EWCS 2024** — job quality and AI-at-work context for the literature review. Manual download.
- **OECD AI and skills reports** — literature review only, not merged into the analysis dataset.
- **Eurostat labour market indicators** — optional extra country-level controls, retrievable automatically the same way as the AI adoption table.

In [6]:
optional_files = list(OPTIONAL_DIR.iterdir()) if OPTIONAL_DIR.exists() else []
print(f'{len(optional_files)} optional supporting file(s) currently present in {OPTIONAL_DIR}')
for f in optional_files:
    print(' -', f.name)

1 optional supporting file(s) currently present in C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\data\raw\optional_sources
 - .gitkeep


## 5. Dataset Inventory

A single structured summary table: source, purpose, country coverage, years, key
variables, access method, status, and limitation for every dataset described above.

In [7]:
dataset_inventory = pd.DataFrame([
    {
        'dataset': 'European Social Survey (ESS)',
        'source': 'ESS Data Portal',
        'purpose': 'Main individual-level dataset',
        'country_coverage': 'ESS participating countries',
        'years': 'Round 11 (2023–2024)',
        'key_variables': 'employment status, age, gender, education, marital status, household size, children/household responsibility, migration background, health, internet use/digital readiness, institutional trust, discrimination, country, design weights',
        'access_method': 'Manual (registration required)',
        'status': 'found' if ess_path else 'missing',
        'limitation': 'Cross-sectional; self-reported measures',
    },
    {
        'dataset': 'ESS Multilevel Data',
        'source': 'ESS Multilevel Data portal',
        'purpose': 'Country-level institutional and macroeconomic context',
        'country_coverage': 'Matched ESS countries',
        'years': 'Matched to ESS round used',
        'key_variables': 'unemployment rate, GDP per capita, Gini, social expenditure, migration indicators, education indicators, institutional indicators',
        'access_method': 'Manual (registration required)',
        'status': 'found' if ess_multilevel_path else 'missing',
        'limitation': 'Indicator availability varies by country/year',
    },
    {
        'dataset': 'Eurostat AI adoption',
        'source': 'Eurostat',
        'purpose': 'Country-level AI adoption context variable',
        'country_coverage': 'EU/EEA countries reporting to Eurostat',
        'years': 'Latest available at time of download',
        'key_variables': 'ai_adoption_enterprises_pct',
        'access_method': 'Automatic (Eurostat API / eurostat package)',
        'status': 'found' if eurostat_files else 'pending',
        'limitation': 'Enterprise-level proxy, not individual-level',
    },
    {
        'dataset': 'Optional supporting sources',
        'source': 'Eurofound EWCS 2024 / OECD / Eurostat labour market indicators',
        'purpose': 'Literature review / optional extra controls',
        'country_coverage': 'Varies',
        'years': 'Varies',
        'key_variables': 'n/a (context/literature) or additional macro controls',
        'access_method': 'Mixed (manual / automatic)',
        'status': 'optional',
        'limitation': 'Not required for core hypotheses',
    },
])

inventory_path = OUTPUTS_TABLES / 'dataset_inventory.xlsx'
dataset_inventory.to_excel(inventory_path, index=False, sheet_name='dataset_inventory')
print('Saved dataset inventory to:', inventory_path)
dataset_inventory

Saved dataset inventory to: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\outputs\tables\dataset_inventory.xlsx


,dataset,source,purpose,country_coverage,years,key_variables,access_method,status,limitation
0,European Social Survey (ESS),ESS Data Portal,Main individual-level dataset,ESS participating countries,Round 11 (2023–2024),"employment status, age, gender, education, mar...",Manual (registration required),found,Cross-sectional; self-reported measures
1,ESS Multilevel Data,ESS Multilevel Data portal,Country-level institutional and macroeconomic ...,Matched ESS countries,Matched to ESS round used,"unemployment rate, GDP per capita, Gini, socia...",Manual (registration required),found,Indicator availability varies by country/year
2,Eurostat AI adoption,Eurostat,Country-level AI adoption context variable,EU/EEA countries reporting to Eurostat,Latest available at time of download,ai_adoption_enterprises_pct,Automatic (Eurostat API / eurostat package),found,"Enterprise-level proxy, not individual-level"
3,Optional supporting sources,Eurofound EWCS 2024 / OECD / Eurostat labour m...,Literature review / optional extra controls,Varies,Varies,n/a (context/literature) or additional macro c...,Mixed (manual / automatic),optional,Not required for core hypotheses


## 6. Status and Next Steps

This notebook only describes and checks the datasets — nothing was cleaned, explored
in depth, or modelled. See `docs/dataset_notes.md` for the narrative version of this
description, kept in sync manually with this notebook.

Next step: `notebooks/02_eda_dataset_understanding.ipynb`. Do not proceed if ESS or
ESS Multilevel data are still missing above.